# Notebook 03 — Does Our Search Really Work?

**VidyaSankalp · Applied GenAI Engineering** · Tutor: Prudhvi Akella

In notebook 02 we asked one question and the results **looked** fine. Today we find out if they **are** fine — by
asking many questions whose right answers we already know, and counting.

```text
0 SETUP            connect, open the answer key, prepare the search
1 THE HOOK         notebook 02 hid a mistake
2 THE ANSWER KEY   questions + their right answers
3 ONE QUESTION     guess, search, pass or fail
4 ALL QUESTIONS    one number for each kind of question
5 THE REAL TEST ★  does the AI get everything it needs?
6 THE MISTAKES ★   why did questions fail? fix one
7 SCORECARD        green or red
8 SAVE THE RUN     MLflow
   Going further   (optional) more scores, languages, and more
```

**You need:** notebook 02 run once, your `.env` file, and `ground_truth_ecom.xlsx` in the `docs` folder.

## 0 — Setup

Six small cells. Run them in order. Each one prints what it did, so you can see that it worked.

In [ ]:
# Install the libraries (only needed once on your computer).
%pip install -q openai pinecone python-dotenv openpyxl mlflow

**0.1 — Connect.** The same keys, model, index and namespace as notebook 02.

In [ ]:
import csv
import math
import os
import re
from datetime import datetime

import openpyxl
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone

# Read the API keys from the .env file.
load_dotenv()

# Stop early with a clear message if a key is missing.
for key in ("OPENAI_API_KEY", "PINECONE_API_KEY"):
    if not os.environ.get(key):
        raise ValueError(f"Please set {key} in your .env file.")

# The OpenAI client turns text into embeddings.
client = OpenAI()

# This MUST be the same model notebook 02 used to store the records.
EMBED_MODEL = "text-embedding-3-small"

# The same index and namespace as notebook 02.
INDEX_NAME = "ecom-kb-tutorial"
NAMESPACE = "ecom-kb"

# Connect to Pinecone and open the index.
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(INDEX_NAME)

print("✅ Connected to", INDEX_NAME)

**0.2 — Open the answer key.** First two small helpers: `read_sheet()` turns a sheet into a list of rows, and
`split_ids()` turns a cell with several ids into a list. Then we read the sheets.

In [ ]:
# Open the Excel file in read-only mode (we only read it).
answer_key = openpyxl.load_workbook("./docs/ground_truth_ecom.xlsx", read_only=True)


def read_sheet(sheet_name):
    """Read one sheet and return its rows as a list of dictionaries.

    Example of one row: {"qid": "T01", "question": "How much did we sell …", "right_id": "glossary::Revenue"}
    """
    rows = answer_key[sheet_name].iter_rows(values_only=True)

    # The first row holds the column names.
    column_names = next(rows)

    records = []
    for values in rows:
        # Skip empty rows.
        if all(value is None for value in values):
            continue

        # Pair each column name with its value in this row.
        record = {}
        for column_name, value in zip(column_names, values):
            record[column_name] = value
        records.append(record)

    return records


def split_ids(cell):
    """Some cells hold several ids, one per line. Turn the cell into a Python list.

    Example: "glossary::Revenue
              column::ecom.orders.total_amount"
          -> ["glossary::Revenue", "column::ecom.orders.total_amount"]
    """
    if not cell:
        return []

    ids = []
    for line in str(cell).split("\n"):
        line = line.strip()
        if line:
            ids.append(line)
    return ids

In [ ]:
# Read every sheet we need into its own list.
glossary_questions = read_sheet("glossary_questions")
example_questions = read_sheet("example_questions")
join_questions = read_sheet("join_questions")
schema_questions = read_sheet("schema_questions")
golden_questions = read_sheet("golden")
language_questions = read_sheet("languages")

print("✅ Answer key opened")
print("   glossary questions:", len(glossary_questions),glossary_questions[0])
print("   example questions: ", len(example_questions),example_questions[0])
print("   join questions:    ", len(join_questions),join_questions[0])
print("   golden questions:  ", len(golden_questions),golden_questions[0])

**0.3 — Check Pinecone.** Get the id of every record notebook 02 stored. If it is empty, notebook 02 wasn't run.

In [ ]:
# index.list() gives the ids in small batches; we collect them all in one set.
ALL_IDS = set()
for id_batch in index.list(namespace=NAMESPACE):
    for record_id in id_batch:
        ALL_IDS.add(record_id)

if len(ALL_IDS) == 0:
    raise ValueError(f"Namespace '{NAMESPACE}' is empty — run notebook 02 first.")

print(f"✅ {len(ALL_IDS)} records found in Pinecone")

**0.4 — Embed every test question once.** Each question becomes 1536 numbers, 100 questions per API call. We keep
them in a dictionary, so every search later can reuse them without calling OpenAI again.

In [ ]:
# STEP 1 — collect every test question (a question that appears twice is kept once).
all_questions = []
for sheet in (glossary_questions, example_questions, join_questions, schema_questions,
              golden_questions, language_questions):
    for row in sheet:
        if row["question"] not in all_questions:
            all_questions.append(row["question"])

# STEP 2 — embed them, 100 per call.
VECTOR = {}                                   # question text -> its 1536 numbers
for start in range(0, len(all_questions), 100):
    batch = all_questions[start : start + 100]
    response = client.embeddings.create(model=EMBED_MODEL, input=batch)

    # The answers come back in the same order as the questions we sent.
    for question, item in zip(batch, response.data):
        VECTOR[question] = item.embedding

print(f"✅ {len(VECTOR)} questions embedded")

**0.5 — The search.** The agent searches three groups of tabs (notebook 02, chapter 8). `search()` runs one of them
for a question and returns the matches, best first.

In [ ]:
# The three groups of tabs — the same as the agent's three lookups in notebook 02.
SCHEMA_TABS = ["nlq_table", "nlq_column", "nlq_join"]       # where the data lives
EXAMPLE_TABS = ["nlq_examples"]                             # saved questions + their SQL
GLOSSARY_TABS = ["glossary", "capability_card"]             # what words mean, and the limits


def search(question, tabs, top_k=10):
    """Search Pinecone for a question, only in the given tabs. Returns the matches, best first.

    We ask for the top 10. The agent only sees the first 3 — places 4 to 10 show how close a miss was.
    """
    results = index.query(
        vector=VECTOR[question],                 # the question's numbers, from step 0.4
        top_k=top_k,
        namespace=NAMESPACE,
        include_metadata=True,
        filter={"sheet": {"$in": tabs}},         # only records from these tabs
    )
    return results.matches


def ids_of(matches):
    """Just the record ids of a list of matches, in the same order."""
    ids = []
    for match in matches:
        ids.append(match.id)
    return ids


# Try it once.
question = "What was our GMV last month?"
print("✅ Search works. Glossary search for the GMV question, top 3:")
for record_id in ids_of(search(question, GLOSSARY_TABS))[:3]:
    print("   ", record_id)

**0.6 — The context pack, as ids.** What the AI receives (notebook 02, chapter 9) comes from three routes:

```text
route 1  FOUND BY SEARCH   top 3 tables/columns + top 2 SQL examples + top 3 glossary/limits
route 2  FOLLOWED LINKS    the columns listed in "maps_to" of the glossary terms that were found
route 3  ALWAYS ADDED      the three capability cards
```

`context_pack()` returns the three routes **separately**, so in chapter 5 we can see which route brought in what.

In [ ]:
ALWAYS_ADDED = ["capability_card::1", "capability_card::2", "capability_card::3"]
EXAMPLES_IN_PACK = 2          # notebook 02 puts the top 2 SQL examples into the pack


def context_pack(question):
    """Run the three searches and return the ids that end up in the context pack, route by route."""

    # The three searches.
    schema_matches = search(question, SCHEMA_TABS)
    example_matches = search(question, EXAMPLE_TABS)
    glossary_matches = search(question, GLOSSARY_TABS)

    # ROUTE 1 — found by search
    found_by_search = []
    found_by_search.extend(ids_of(schema_matches)[:3])
    found_by_search.extend(ids_of(example_matches)[:EXAMPLES_IN_PACK])
    found_by_search.extend(ids_of(glossary_matches)[:3])

    # ROUTE 2 — followed links: for each glossary term found, add the columns in its "maps_to"
    followed_links = []
    for match in glossary_matches[:3]:
        if match.metadata.get("sheet") == "glossary":
            for column in match.metadata.get("maps_to", []):
                column_id = "column::" + column           # "ecom.orders.total_amount" -> "column::ecom.orders.total_amount"
                if column_id in ALL_IDS:
                    followed_links.append(column_id)

    # ROUTE 3 — always added
    always_added = ALWAYS_ADDED

    # The best score of all three searches (used in "Going further", part C).
    best_score = max(schema_matches[0].score, example_matches[0].score, glossary_matches[0].score)

    return {
        "found_by_search": found_by_search,
        "followed_links": followed_links,
        "always_added": always_added,
        "glossary_top": ids_of(glossary_matches),
        "best_score": best_score,
    }

In [ ]:
# Try it once on the GMV question.
pack = context_pack("What was our GMV last month?")
print("✅ Context pack works. For the GMV question:")
print("   route 1 — found by search:", len(pack["found_by_search"]), "records")
print("   route 2 — followed links: ", len(pack["followed_links"]), "records")
print("   route 3 — always added:   ", len(pack["always_added"]), "records")

## 1 — The hook: notebook 02 hid a mistake

In notebook 02 we asked **"What was our GMV last month?"** The answer needs the column `orders.total_amount` — that
is where revenue is stored.

**Guess first:** when we search the **tables and columns** for this question, will `orders.total_amount` be in the
top 3? Then run the cell.

In [ ]:
question = "What was our GMV last month?"

# Search only the tables and columns, and keep the top 3 (what the agent sees).
top_3 = ids_of(search(question, SCHEMA_TABS))[:3]

print("Question:", question)
print("Top 3 tables/columns found:")
for record_id in top_3:
    print("   ", record_id)

# Is the column we need among them?
print()
if "column::ecom.orders.total_amount" in top_3:
    print("✅ orders.total_amount was found")
else:
    print("❌ orders.total_amount was NOT found — the search missed the most important column")

In the real run, the search **missed** it. Notebook 02 still gave the AI the right column — through the glossary's
link (Revenue → `orders.total_amount`). But we only know that because we **looked closely at one question**.

Users will ask thousands of different questions. We can't look at each one by hand. So we need a list of questions
**with the right answers written down**, and a way to **count**.

## 2 — The answer key

`docs/ground_truth_ecom.xlsx` is our answer key: test questions, and for each one the **id** of the record that
should be found. Open it in Excel if you like. Here are three rows:

In [ ]:
# Show the first 3 rows of the glossary questions.
for row in glossary_questions[:3]:
    print("Question:    ", row["question"])
    print("Right answer:", row["right_id"])
    print()

Notice: the first question asks about **"merchandise, before delivery charges and tax"** — it never says
"Revenue". That is on purpose.

**The one rule:** a test question must never be the same text we stored. Otherwise the search finds it by simply
copying — like an exam where the questions are printed in the textbook. Our questions use **other words with the
same meaning**, the way real users talk.

| Sheet | How many | Example |
|---|---:|---|
| `glossary_questions` | 66 | "How much does a typical checkout bring in?" → Average Order Value |
| `example_questions` | 61 | "Which ten products sold the most pieces?" → a saved SQL example |
| `join_questions` | 16 | "Which warehouse city sends the most parcels?" → a join between two tables |
| `golden` | 49 | real business questions, like a manager would ask |

## 3 — One question: guess, search, pass or fail

The AI only ever sees the **top 3** search results. So our rule is simple:

```text
right answer in the top 3?   ✅ PASS
not in the top 3?            ❌ FAIL
```

**Guess first:** *"How much does a typical checkout bring in?"* — the right answer is `glossary::Average Order Value`.
The question doesn't use any of those words. Will the search still find it?

In [ ]:
# Pick one question. Change the number (0 to 65) to try other questions.
row = glossary_questions[2]
question = row["question"]
right_answer = row["right_id"]

# Search the glossary tabs and keep the top 3.
top_3 = ids_of(search(question, GLOSSARY_TABS))[:3]

print("Question:    ", question)
print("Right answer:", right_answer)
print()

# Show the top 3 and mark the right answer if it is there.
place = 1
for record_id in top_3:
    if record_id == right_answer:
        print(f"  {place}. {record_id}   ← right answer")
    else:
        print(f"  {place}. {record_id}")
    place += 1

# Pass or fail?
print()
if right_answer in top_3:
    print("✅ PASS")
else:
    print("❌ FAIL")

**Try it:** change `glossary_questions[2]` to another number between 0 and 65 and run the cell again. Guess before
each run. Some will pass, some will fail.

Doing this 66 times by hand would be boring. Let the computer do it.

## 4 — All questions: one number for each kind

The same check — "is the right answer in the top 3?" — for every question. Then we count:

```text
passed 50 of 66   →   50 ÷ 66 = 0.76      (1.00 would mean every question passed)
```

First the two helper functions, then the check.

In [ ]:
def check_all(sheet_name, questions, tabs):
    """Check every question of one sheet: is its right answer in the top 3?

    Returns one result (a dictionary) per question.
    """
    results = []
    for row in questions:
        # Search, and keep the top 10 (we look at the first 3; the rest is used in "Going further").
        top_10 = ids_of(search(row["question"], tabs))

        # A few questions have more than one right answer; finding any one of them counts.
        right_answers = split_ids(row["right_id"])

        # 1 = passed, 0 = failed
        passed = 0
        for record_id in top_10[:3]:
            if record_id in right_answers:
                passed = 1

        results.append({
            "set": sheet_name,
            "qid": row["qid"],
            "question": row["question"],
            "right_answer": " | ".join(right_answers),
            "top_3": " | ".join(top_10[:3]),
            "top_10": top_10,
            "right_answers": right_answers,
            "passed": passed,
        })
    return results


def count_passed(results):
    """How many results passed."""
    passed = 0
    for result in results:
        if result["passed"] == 1:
            passed += 1
    return passed

In [ ]:
# Check every question of three sheets (this takes about a minute).
glossary_results = check_all("glossary", glossary_questions, GLOSSARY_TABS)
example_results = check_all("examples", example_questions, EXAMPLE_TABS)
join_results = check_all("joins", join_questions, SCHEMA_TABS)

# One line per sheet: passed X of Y = score.
print("Right answer in the top 3:")
for name, results in (("glossary", glossary_results), ("examples", example_results), ("joins", join_results)):
    passed = count_passed(results)
    total = len(results)
    print(f"  {name:9} passed {passed:3} of {total:3}  =  {passed / total:.2f}")

**What to notice:**

- **examples** usually scores high: example questions are written the way users talk.
- **glossary** is harder: our questions never use the business word, so the search must understand the **meaning**.
- **joins** is usually lowest: every join text starts with "Join A to B on …", so they all look alike.

> **But:** the AI doesn't get just one record. It gets a whole **context pack**. Does the pack have everything?

## 5 — The real test: does the AI get everything it needs? ★

Each `golden` question lists the records the AI **must** have. The question passes only if **all** of them are in
the context pack.

```text
"What was our GMV last month?"
   must have:   glossary::Revenue                    what GMV means here
                column::ecom.orders.total_amount     where the number is stored

both in the pack?   ✅ PASS      2 of 2 = 1.0
only one?           ❌ FAIL      1 of 2 = 0.5   ← this number is the "pack completeness"
```

**Guess first:** for the GMV question — which route brings in `orders.total_amount`?

In [ ]:
question = "What was our GMV last month?"

# What the answer key says this question must have.
must_have = split_ids(golden_questions[0]["must_find"])

# Build the context pack (as ids) for the question.
pack = context_pack(question)

# For each needed record: which route brought it in — or is it missing?
for record_id in must_have:
    if record_id in pack["found_by_search"]:
        how = "✅ found by search"
    elif record_id in pack["followed_links"]:
        how = "✅ followed link"
    elif record_id in pack["always_added"]:
        how = "✅ always added"
    else:
        how = "❌ MISSING"
    print(f"{record_id:40} {how}")

Now all the golden questions the AI should answer. We count twice — **without** the links and **with** them — to see
what the links from notebook 02 are worth. First a small helper, then the check.

In [ ]:
def share_in(needed, pack_ids):
    """What share of the needed ids is in the pack.  2 of 2 -> 1.0   1 of 2 -> 0.5   0 of 2 -> 0.0"""
    found = 0
    for record_id in needed:
        if record_id in pack_ids:
            found += 1
    return found / len(needed)


def average(results, key):
    """The average of one value (like "completeness") over a list of results."""
    total = 0
    for result in results:
        total += result[key]
    return total / len(results)

In [ ]:
golden_results = []

for row in golden_questions:
    must_have = split_ids(row["must_find"])

    # Skip the questions the AI should refuse — they are in "Going further", part C.
    if row["behaviour"] == "refuse" or len(must_have) == 0:
        continue

    # STEP 1 — build the pack two ways: search only, and the real pack (search + links + always added).
    pack = context_pack(row["question"])
    search_only = pack["found_by_search"]
    real_pack = pack["found_by_search"] + pack["followed_links"] + pack["always_added"]

    # STEP 2 — which needed records are missing from the real pack?
    missing = []
    for record_id in must_have:
        if record_id not in real_pack:
            missing.append(record_id)

    # STEP 3 — keep the result of this question.
    if len(missing) == 0:
        passed = 1
    else:
        passed = 0

    golden_results.append({
        "set": "golden",
        "qid": row["qid"],
        "question": row["question"],
        "type": row["type"],
        "search_only": share_in(must_have, search_only),
        "completeness": share_in(must_have, real_pack),
        "missing": " | ".join(missing),
        "top_3": " | ".join(pack["glossary_top"][:3]),
        "passed": passed,
    })

In [ ]:
# The averages over all business questions.
print(f"{len(golden_results)} business questions")
print(f"  pack completeness — search only:   {average(golden_results, 'search_only'):.2f}")
print(f"  pack completeness — with links:    {average(golden_results, 'completeness'):.2f}")
print(f"  questions with EVERYTHING needed:  {count_passed(golden_results)} of {len(golden_results)}")

**What to notice:** the jump from "search only" to "with links" is what the `maps_to` links from notebook 02 are
worth — records the search missed, but the links still brought in. That design choice is now **measured**, not
just argued.

> **Next:** which questions failed, and why?

## 6 — The mistakes ★

A score tells you **how much** is wrong. The mistakes tell you **what**. Here are five failed business questions.
For each one, **guess why** before reading the table below.

In [ ]:
# Show the first 5 business questions that failed.
shown = 0
for result in golden_results:
    if result["passed"] == 0 and shown < 5:
        print(f"{result['qid']}  {result['question']}")
        print(f"     missing:               {result['missing']}")
        print(f"     glossary search top 3: {result['top_3']}")
        print()
        shown += 1

In [ ]:
# Save EVERY mistake to a CSV file you can open in Excel.
os.makedirs("eval_runs", exist_ok=True)

# STEP 1 — collect the failed questions of all four checks.
mistakes = []
for results in (golden_results, glossary_results, example_results, join_results):
    for result in results:
        if result["passed"] == 0:
            mistakes.append(result)

# STEP 2 — write them to eval_runs/misses.csv (only these columns).
columns = ["set", "qid", "question", "right_answer", "missing", "top_3"]
with open("eval_runs/misses.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(mistakes)

print(f"{len(mistakes)} mistakes saved to eval_runs/misses.csv")

Almost every mistake is one of three kinds:

| What you see | Why | Fix |
|---|---|---|
| the user's word (like "OTD", "BOPIS") isn't in the record | the stored text doesn't know that word | add it to the term's **synonyms** in `kb_feed_ecom.xlsx` |
| the glossary term was found, but its column is missing | the link is missing | add the column to the term's **maps_to** |
| a join is missing (question about several tables) | similar is not the same as connected | the agent follows joins in code (Agent V3) |

**The good news:** most fixes are in the **Excel file**, not in the code.

**Try it:** pick one mistake. Fix it in `kb_feed_ecom.xlsx`. Run notebook 02 again (to reload), then this notebook.
Did it pass?

## 7 — Scorecard

We decide what is "good enough" **before** looking at the results — otherwise we'd just pick a number below
whatever we got.

In [ ]:
# STEP 1 — the targets, decided before the run.
TARGETS = {
    "business questions: pack completeness": 0.90,
    "glossary: in top 3": 0.80,
    "examples: in top 3": 0.80,
    "joins: in top 3": 0.60,      # joins all look alike, so we accept less
}

# STEP 2 — this run's scores.
SCORES = {
    "business questions: pack completeness": average(golden_results, "completeness"),
    "glossary: in top 3": count_passed(glossary_results) / len(glossary_results),
    "examples: in top 3": count_passed(example_results) / len(example_results),
    "joins: in top 3": count_passed(join_results) / len(join_results),
}

# STEP 3 — compare: green if the score reaches the target, red if not.
targets_met = 0
for name in TARGETS:
    score = SCORES[name]
    target = TARGETS[name]
    if score >= target:
        targets_met += 1
        print(f"✅ {name:40} {score:.2f}   (target {target:.2f})")
    else:
        print(f"❌ {name:40} {score:.2f}   (target {target:.2f})")

print()
print(f"{targets_met} of {len(TARGETS)} targets met")

A ❌ is not a failure — it is your **to-do list**. Open `eval_runs/misses.csv`, fix the knowledge, run again.

## 8 — Save the run

Each time you run this notebook you get new scores. **MLflow** saves every run, so you can compare them side by side
in your browser — for example before and after a fix. It runs on your computer; no account needed.
See `README_MLflow.md` for how to open it.

```text
MLflow run
 ├── parameters   the settings used        embed_model, examples_in_pack
 ├── metrics      the scores               the 4 scorecard numbers + targets met
 ├── table        every question           passed 1 or 0
 └── artifact     a file                   misses.csv
```

In [ ]:
import mlflow

# Save runs in a file called mlflow.db, next to this notebook.
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# All runs of this notebook go into one "experiment" (like a folder).
mlflow.set_experiment("vectordb-03-retrieval-eval")

# The run's name: the date and time, e.g. run_20261117_142621.
run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")

# MLflow names may only use letters, numbers, _ - . / and spaces — so we give each score a simple name.
METRIC_NAMES = {
    "business questions: pack completeness": "golden_pack_completeness",
    "glossary: in top 3": "glossary_in_top_3",
    "examples: in top 3": "examples_in_top_3",
    "joins: in top 3": "joins_in_top_3",
}

In [ ]:
with mlflow.start_run(run_name=run_name):

    # STEP 1 — parameters: the settings used for this run
    mlflow.log_param("embed_model", EMBED_MODEL)
    mlflow.log_param("examples_in_pack", EXAMPLES_IN_PACK)

    # STEP 2 — metrics: the scores from the scorecard
    for name in SCORES:
        mlflow.log_metric(METRIC_NAMES[name], SCORES[name])
    mlflow.log_metric("golden_search_only", average(golden_results, "search_only"))
    mlflow.log_metric("targets_met", targets_met)

    # STEP 3 — a table with one row per question: was it passed?
    table = {"set": [], "qid": [], "question": [], "passed": []}
    for results in (golden_results, glossary_results, example_results, join_results):
        for result in results:
            table["set"].append(result["set"])
            table["qid"].append(result["qid"])
            table["question"].append(result["question"])
            table["passed"].append(result["passed"])
    mlflow.log_table(data=table, artifact_file="per_question.json")

    # STEP 4 — artifact: attach the mistakes file
    mlflow.log_artifact("eval_runs/misses.csv")

print("✅ Saved to MLflow as", run_name)
print("To look at it: mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001   (see README_MLflow.md)")

## What you learned

Looking at one question isn't proof → we wrote down the **right answers first** → we checked hundreds of questions
and **counted** → the AI got most of what it needed, and the **links** added a measurable part → the **mistakes**
showed what to fix, mostly in the Excel file → every run is **saved** to compare.

## Quick check

1. Why isn't one good-looking question enough?
2. Why must a test question never be the exact text we stored?
3. What does "in top 3 = 0.76" mean?
4. What is pack completeness?
5. Most mistakes are fixed where — in the code or in the Excel file?

<details><summary><b>Answers</b> (click)</summary>

1. It is only one example — the GMV search missed a column, and only many questions show how often that happens.
2. The search would find it by copying, not by understanding.
3. For 76% of the questions, the right answer was in the top 3 results.
4. Of the records the AI must have, the share that is in the context pack (2 of 2 = 1.0).
5. In the Excel file: a missing synonym or a missing `maps_to` link.

</details>

**Next:** Agent Version 3 uses these searches as tools — and checks the last part: the SQL and the answer.

---

# Going further (optional)

The main part is done. Each part below is short and stands alone — pick what interests you.

### A — Is 1st place better than 3rd? (rank score)

"In top 3" treats 1st and 3rd place the same. Two scores that don't:

```text
found first (Hit@1)   1 if the right answer is in 1st place, else 0
rank score  (MRR)     1 ÷ the place:   1st → 1.0    2nd → 0.5    3rd → 0.33    not found → 0
```

In [ ]:
def rank_score(top_ids, right_answers):
    """1 / the place of the first right answer.  1st -> 1.0, 2nd -> 0.5, 3rd -> 0.33, not found -> 0."""
    place = 1
    for record_id in top_ids:
        if record_id in right_answers:
            return 1 / place
        place += 1
    return 0


def found_first(top_ids, right_answers):
    """1 if the right answer is in 1st place, else 0."""
    if len(top_ids) > 0 and top_ids[0] in right_answers:
        return 1
    return 0


# Average both scores over each sheet.
for name, results in (("glossary", glossary_results), ("examples", example_results), ("joins", join_results)):
    total_rank = 0
    total_first = 0
    for result in results:
        total_rank += rank_score(result["top_10"], result["right_answers"])
        total_first += found_first(result["top_10"], result["right_answers"])
    print(f"{name:9}  found first {total_first / len(results):.2f}   rank score {total_rank / len(results):.2f}")

**nDCG** goes one step further: some records matter more (must-have = 2 points, helpful = 1 point), and points
count less the lower they are in the list (`points ÷ log2(place + 1)`). nDCG = our points ÷ the best possible
points — 1.0 when the most important records are at the top.

### B — Is the embedding model better than counting words?

The simplest possible search: count the words the question and the record share. If our embedding search doesn't
beat this, the model isn't worth its cost.

In [ ]:
# STEP 1 — read the stored text of every record back from Pinecone, 100 at a time.
stored_text = {}
stored_sheet = {}
all_ids_list = sorted(ALL_IDS)
for start in range(0, len(all_ids_list), 100):
    fetched = index.fetch(ids=all_ids_list[start : start + 100], namespace=NAMESPACE)
    for record_id, record in fetched.vectors.items():
        stored_text[record_id] = record.metadata.get("text", "")
        stored_sheet[record_id] = record.metadata.get("sheet", "")

print(len(stored_text), "record texts read")

In [ ]:
# Small words we ignore when counting.
SMALL_WORDS = {"a", "an", "the", "of", "to", "in", "on", "for", "by", "with", "and", "or", "is", "are", "was", "do",
               "does", "did", "we", "our", "us", "how", "what", "which", "who", "when", "where", "much", "many",
               "each", "per", "from", "at", "it", "this", "that", "have", "has", "more", "most", "not"}


def words_of(text):
    """The meaningful words of a text, in lower case."""
    words = set()
    for word in re.findall(r"[a-z0-9]+", str(text).lower()):
        if word not in SMALL_WORDS:
            words.add(word)
    return words


def word_search_top_3(question, tabs):
    """Rank the records by how many words they share with the question; return the top 3 ids."""
    question_words = words_of(question)

    # STEP 1 — count the shared words for every record in these tabs.
    scored = []
    for record_id in stored_text:
        if stored_sheet[record_id] in tabs:
            shared = 0
            for word in words_of(stored_text[record_id]):
                if word in question_words:
                    shared += 1
            scored.append((shared, record_id))

    # STEP 2 — sort: most shared words first.
    scored.sort(reverse=True)

    # STEP 3 — keep the ids of the top 3.
    top_3 = []
    for shared, record_id in scored[:3]:
        top_3.append(record_id)
    return top_3

In [ ]:
# The same questions, the same tabs, the same rule (right answer in the top 3).
print("In the top 3:  word search   embeddings")
for name, questions, results, tabs in (("glossary", glossary_questions, glossary_results, GLOSSARY_TABS),
                                       ("examples", example_questions, example_results, EXAMPLE_TABS)):
    word_passed = 0
    for row in questions:
        if row["right_id"] in word_search_top_3(row["question"], tabs):
            word_passed += 1
    print(f"  {name:10} {word_passed / len(questions):10.2f} {count_passed(results) / len(results):12.2f}")

### C — Questions the AI should refuse

Some golden questions can't be answered from our data ("How many people visited our website yesterday?") or have
nothing to do with it ("Who won the cricket match?"). Does the search find the "WHAT IT CANNOT ANSWER" card?

In [ ]:
for row in golden_questions:
    if row["behaviour"] != "refuse":
        continue

    pack = context_pack(row["question"])

    # Is the "WHAT IT CANNOT ANSWER" card in the glossary search's top 3?
    if "capability_card::2" in pack["glossary_top"][:3]:
        mark = "✅ limits card found"
    else:
        mark = "—  limits card not in top 3"

    print(f"{row['type']:13} best score {pack['best_score']:.2f}   {mark:28} {row['question']}")

Compare these **best scores** with those of real questions (usually higher, but not always). A score cut-off alone
would also throw away real questions — that's why the AI always gets the capability cards and decides itself.

### D — Telugu and Hinglish

The same business questions, in Telugu and Hinglish. Do they still get what they need?

In [ ]:
# Find a golden question by its id (G01, G03 …).
golden_by_id = {}
for row in golden_questions:
    golden_by_id[row["qid"]] = row

for row in language_questions:
    # The right answers are the same as for the English question.
    must_have = split_ids(golden_by_id[row["qid"]]["must_find"])

    pack = context_pack(row["question"])
    real_pack = pack["found_by_search"] + pack["followed_links"] + pack["always_added"]

    print(f"{row['qid']}  {row['language']:9} completeness {share_in(must_have, real_pack):.2f}   {row['question']}")

### E — Schema questions: easy vs real wording

`schema_questions` come from the workbook. Most of them contain the column's own name ("How many orders per
payment method?") — easy for any search. Only the real-wording ones are a fair test.

In [ ]:
# Split the schema questions into two groups.
easy = []
real = []
for row in schema_questions:
    if row["names_the_record"] == "Y":
        easy.append(row)
    else:
        real.append(row)

# Check both groups the same way as chapter 4 (about a minute: 582 searches).
for name, questions in (("names the column", easy), ("real wording", real)):
    results = check_all("schema", questions, SCHEMA_TABS)
    print(f"{name:17} in top 3: {count_passed(results) / len(results):.2f}   ({len(results)} questions)")

### F — What changed since the last run?

After you fix something and run again, this shows which questions changed — so you also catch a fix that broke
something else.

In [ ]:
# STEP 1 — this run's result per question, under a unique key like "golden G03".
this_run = {}
for results in (golden_results, glossary_results, example_results, join_results):
    for result in results:
        key = result["set"] + " " + str(result["qid"])
        this_run[key] = result

# STEP 2 — save this run to a CSV.
run_file = f"eval_runs/{run_name}.csv"
with open(run_file, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["key", "question", "passed"])
    for key in this_run:
        writer.writerow([key, this_run[key]["question"], this_run[key]["passed"]])
print("Saved", run_file)

In [ ]:
# STEP 3 — find the run saved before this one.
earlier_runs = []
for file_name in sorted(os.listdir("eval_runs")):
    if file_name.startswith("run_") and file_name != run_name + ".csv":
        earlier_runs.append(file_name)

if len(earlier_runs) == 0:
    print("This is the first run — fix something, run again, then come back to this cell.")
else:
    previous_file = "eval_runs/" + earlier_runs[-1]

    # STEP 4 — read whether each question passed last time.
    passed_before = {}
    with open(previous_file, encoding="utf-8") as file:
        for row in csv.DictReader(file):
            passed_before[row["key"]] = row["passed"]

    # STEP 5 — print every question whose result changed.
    changes = 0
    for key in this_run:
        if key in passed_before and passed_before[key] != str(this_run[key]["passed"]):
            changes += 1
            if this_run[key]["passed"] == 1:
                print(f"✅ now passes   {key}   {this_run[key]['question']}")
            else:
                print(f"❌ now fails    {key}   {this_run[key]['question']}")

    print(f"\n{changes} questions changed since {earlier_runs[-1]}")